# Async Programming — Recap

Why this matters: from Module 7 onward, tool calls and the Module 12 MCP client are `async`. This notebook is the minimum you need to *read* that code comfortably.

## 1. Synchronous programming — one thing after another

In [1]:
import time

def fetch_user_profile():
    print("fetching user profile...")
    time.sleep(2)
    print("user profile fetched")

def fetch_order_history():
    print("fetching order history...")
    time.sleep(3)
    print("order history fetched")

def fetch_recommendations():
    print("fetching recommendations...")
    time.sleep(1)
    print("recommendations fetched")

def main_sync():
    start = time.time()
    fetch_user_profile()
    fetch_order_history()
    fetch_recommendations()
    print(f"Total time (sync): {time.time() - start:.1f}s")

main_sync()

fetching user profile...


user profile fetched
fetching order history...


order history fetched
fetching recommendations...


recommendations fetched
Total time (sync): 6.0s


Three unrelated fetches, run one after another. Expect roughly **6 seconds** — the *sum* of the three waits.

## 2. Asynchronous programming — same work, run concurrently

In [2]:
import asyncio

async def fetch_user_profile():
    print("fetching user profile...")
    await asyncio.sleep(2)
    print("user profile fetched")

async def fetch_order_history():
    print("fetching order history...")
    await asyncio.sleep(3)
    print("order history fetched")

async def fetch_recommendations():
    print("fetching recommendations...")
    await asyncio.sleep(1)
    print("recommendations fetched")

async def main_async():
    start = time.time()
    await asyncio.gather(
        fetch_user_profile(),
        fetch_order_history(),
        fetch_recommendations(),
    )
    print(f"Total time (async): {time.time() - start:.1f}s")

# Jupyter already runs an event loop, so you can `await` directly in a cell
# (plain scripts need asyncio.run(main_async()) instead).
await main_async()

fetching user profile...
fetching order history...
fetching recommendations...


recommendations fetched


user profile fetched


order history fetched
Total time (async): 3.0s


Same three waits, but they overlap. Expect roughly **3 seconds** — the *slowest* one, not the sum. That's the entire point of `async`/`await`.

## 3. The one mistake you will make: forgetting `await`

In [3]:
async def fetch_recommendations():
    await asyncio.sleep(1)
    return "recommendations"

result = fetch_recommendations()   # missing await
print("without await:", result)    # a <coroutine object>, not the string

result = await fetch_recommendations()
print("with await:", result)

without await: <coroutine object fetch_recommendations at 0x109e1e2c0>


with await: recommendations


/var/folders/jl/653x1f1n6px6ntjl_l6g6zt00000gn/T/ipykernel_81126/3324619420.py:8: RuntimeWarning: coroutine 'fetch_recommendations' was never awaited
  result = await fetch_recommendations()


Calling an `async def` function does **nothing** until you `await` it — you just get a coroutine object back. This is exactly the bug in Module 12 when `agent.invoke(...)` is used where `ainvoke` was needed.

## 4. Concurrency vs. parallelism

- **Parallelism** — tasks literally running at the same time, usually on multiple CPU cores.
- **Concurrency** — tasks *taking turns*: one pauses at `await` so another can run, usually on a single core.

`asyncio` gives you concurrency, not parallelism. That's ideal for I/O waits (network calls, API calls) — it does nothing for CPU-heavy work.

## 5. Why this matters for agents

In [4]:
# Two independent "agent calls" that don't depend on each other.
# Running them concurrently cuts total wait time — the same idea behind
# Module 5's Send and Module 7's parallel tool calls.

async def call_agent(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{name} finished"

async def run_agents():
    results = await asyncio.gather(
        call_agent("research_agent", 2),
        call_agent("summary_agent", 1),
    )
    print(results)

await run_agents()

['research_agent finished', 'summary_agent finished']
